# RLVR with a Custom Reward Function

## Lab 3 - Train with `RLVRTrainer`

This notebook launches a SageMaker serverless RLVR customization job using the GSM8K datasets from notebook 1 and the custom Reward Function evaluator from notebook 2.

## 1. Set up SageMaker

In [ ]:
import boto3
import os
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix
project_prefix = "gsm8k-custom-reward-rlvr"
reward_function_name = "gsm8k-rlvr-rf"

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sess.boto_region_name}")

### Choosing a base model

The base model used across this lab is configured in [`config.py`](config.py). Want to try a different model? You can browse the available models on SageMaker JumpStart by running the cell below. To switch models, just update `BASE_MODEL_ID` in `config.py` — all notebooks in this lab will pick up the change automatically.

> **Note:** Not every JumpStart model supports every customization technique (SFT, DPO, RLVR, etc.). If you hit a "No recipes found" error, that model may not support the technique used in this lab.

In [ ]:
import boto3

# Retrieve all JumpStart models that support customization (fine-tuning)
sm = boto3.client("sagemaker")
models = []
kwargs = {"HubName": "SageMakerPublicHub", "HubContentType": "Model", "MaxResults": 100}
while True:
    response = sm.list_hub_contents(**kwargs)
    for item in response["HubContentSummaries"]:
        keywords = item.get("HubContentSearchKeywords", [])
        if "@capability:customization" in keywords:
            models.append(item["HubContentName"])
    if "NextToken" in response:
        kwargs["NextToken"] = response["NextToken"]
    else:
        break

models.sort()
print(f"Available models ({len(models)}):")
print("\n".join(models))

## 2. Retrieve datasets and reward function evaluator

In [ ]:
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.evaluator import Evaluator

base_model_id = "huggingface-reasoning-qwen3-06b"
training_dataset = DataSet.get(name=f"{project_prefix}-train")
validation_dataset = DataSet.get(name=f"{project_prefix}-val")
custom_reward_function = Evaluator.get(name=reward_function_name)

job_name = f"custom-reward-rlvr-{base_model_id.split('/')[-1].replace('.', '-')}"
mlflow_experiment_name = "qwen3-06b-custom-reward-rlvr"

if default_prefix:
    output_path = f"s3://{bucket_name}/{default_prefix}/{base_model_id}-custom-reward-rlvr"
else:
    output_path = f"s3://{bucket_name}/{base_model_id}-custom-reward-rlvr"

os.environ["SAGEMAKER_MLFLOW_CUSTOM_ENDPOINT"] = (
    f"https://mlflow.sagemaker.{sess.boto_region_name}.app.aws"
)

print(f"Base model:             {base_model_id}")
print(f"Training data:          {training_dataset.arn}")
print(f"Validation data:        {validation_dataset.arn}")
print(f"Reward function:        {custom_reward_function.arn}")
print(f"Output path:            {output_path}")

## 3. Create a model package group

In [ ]:
import hashlib
from botocore.exceptions import ClientError
from sagemaker.core.resources import ModelPackageGroup

MAX_MPG_NAME_LENGTH = 63
suffix = "-custom-rw-rlvr"

candidate = f"{base_model_id}{suffix}"
if len(candidate) > MAX_MPG_NAME_LENGTH:
    digest = hashlib.sha1(base_model_id.encode()).hexdigest()[:6]
    # reserve room for the suffix, a hyphen separator, and the 6-char hash
    keep = MAX_MPG_NAME_LENGTH - len(suffix) - len(digest) - 1
    truncated = base_model_id[:keep].rstrip("-")
    model_package_group_name = f"{truncated}-{digest}{suffix}"
else:
    model_package_group_name = candidate

try:
    model_package_group = ModelPackageGroup.get(
        model_package_group_name=model_package_group_name
    )
    print(f"Model Package Group already exists: {model_package_group_name}")
except ClientError:
    model_package_group = ModelPackageGroup.create(
        model_package_group_name=model_package_group_name,
        model_package_group_description="Store models from SageMaker serverless RLVR customization with custom reward function",
    )
    print(f"Created Model Package Group: {model_package_group_name}")

## 4. Configure `RLVRTrainer`

The SageMaker Python SDK v3 `RLVRTrainer` accepts a custom Reward Function evaluator through the `custom_reward_function` parameter.

In [ ]:
from sagemaker.train.rlvr_trainer import RLVRTrainer

trainer = RLVRTrainer(
    model=base_model_id,
    model_package_group=model_package_group,
    custom_reward_function=custom_reward_function,
    training_dataset=training_dataset,
    validation_dataset=validation_dataset,
    s3_output_path=output_path,
    mlflow_experiment_name=mlflow_experiment_name,
    base_job_name=job_name,
    sagemaker_session=sess,
    accept_eula=True,
    role=role,
)

### Inspect and adjust hyperparameters

In [ ]:
from rich.pretty import pprint

print("Default finetuning options:")
pprint(trainer.hyperparameters.to_dict())

In [ ]:
trainer.hyperparameters.max_epochs = 1
trainer.hyperparameters.learning_rate = 5e-06
trainer.hyperparameters.global_batch_size = 128
trainer.hyperparameters.max_prompt_length = 1024

## 5. Start training

In [ ]:
training_job = trainer.train(wait=False)
print(f"Training job: {training_job.training_job_name}")